# Baseline de regresión

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_train = np.load("data/X_train_processed.npy")
X_val = np.load("data/X_val_processed.npy")
y_train = np.load("data/y_train_usd.npy")
y_val = np.load("data/y_val_usd.npy")
print(X_train.shape, X_val.shape)

(934, 278) (234, 278)


In [2]:
def metrics(y_true, pred):
    return {
        "RMSE_USD": mean_squared_error(y_true, pred) ** 0.5,
        "MAE_USD": mean_absolute_error(y_true, pred),
        "R2": r2_score(y_true, pred),
    }

results=[]
models={}

lr = LinearRegression().fit(X_train, y_train)
pred = lr.predict(X_val)
results.append({"modelo":"LinearRegression", **metrics(y_val,pred)})
models["LinearRegression"] = lr

for alpha in [0.1, 1.0, 10.0, 30.0, 100.0]:
    m=Ridge(alpha=alpha).fit(X_train,y_train)
    p=m.predict(X_val)
    results.append({"modelo":f"Ridge alpha={alpha}", **metrics(y_val,p)})
    models[f"Ridge alpha={alpha}"]=m

results_df=pd.DataFrame(results).sort_values("RMSE_USD").reset_index(drop=True)
results_df

,modelo,RMSE_USD,MAE_USD,R2
0,Ridge alpha=100.0,31360.028571,17295.587891,0.820453
1,Ridge alpha=30.0,31379.978330,16878.876953,0.820225
2,Ridge alpha=10.0,32085.873776,17397.869141,0.812046
3,Ridge alpha=1.0,36495.309068,18791.769531,0.756836
4,Ridge alpha=0.1,42351.132594,19916.500000,0.672543
5,LinearRegression,44359.818936,20586.380859,0.640744


In [3]:
best_name = results_df.iloc[0]["modelo"]
best_rmse = results_df.iloc[0]["RMSE_USD"]
print("Mejor baseline:", best_name)
print(f"RMSE: ${best_rmse:,.2f}")
joblib.dump(models[best_name], "models/best_baseline_v2.joblib")
results_df.to_csv("reports/baseline_v2.csv", index=False)

Mejor baseline: Ridge alpha=100.0
RMSE: $31,360.03
